In [1]:
# Importing required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Notebook formatting
pd.set_option('display.max_columns', None)

# Importing harmless warnings
import warnings
warnings.filterwarnings('ignore')

#NLP - Natural language processing
import nltk  #natural language tool-kit
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

#Importing required module from sklearn library
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Importing datas
movies = pd.read_csv('movies_metadata.csv', dtype={10: str})
credits = pd.read_csv('credits.csv')
keywords = pd.read_csv('keywords.csv')

In [3]:
# Movies Sample
movies.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",3.859495,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,8.387519,/e64sOI48hQXyru7naBFyssKFxVd.jpg,"[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [4]:
# Credits sample
credits.head()

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862


In [5]:
# Keywords sample
keywords.head()

,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."
3,31357,"[{'id': 818, 'name': 'based on novel'}, {'id':..."
4,11862,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n..."


In [6]:
# Data Volumes
print(f'Movies Shape: {movies.shape}')
print(f'Credits Shape: {credits.shape}')
print(f'Keywords Shape: {keywords.shape}')

Movies Shape: (45466, 24)
Credits Shape: (45476, 3)
Keywords Shape: (46419, 2)


In [7]:
# Merging all the 3 datasets

# Changing Dtype
movies['id'] = movies['id'].astype(str)
credits['id'] = credits['id'].astype(str)
keywords['id'] = keywords['id'].astype(str)

# Merging
movies = movies.merge(credits, on='id').merge(keywords, on='id')

In [8]:
# Data sample after merging
movies.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count,cast,crew,keywords
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...","[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...","[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",3.859495,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...","[{'id': 818, 'name': 'based on novel'}, {'id':..."
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,8.387519,/e64sOI48hQXyru7naBFyssKFxVd.jpg,"[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...","[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n..."


In [9]:
# Required features
req_movies = movies[['genres', 'id', 'overview', 'poster_path', 'title', 'cast', 'crew', 'keywords']]
req_movies

,genres,id,overview,poster_path,title,cast,crew,keywords
0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",862,"Led by Woody, Andy's toys live happily in his ...",/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,Toy Story,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",8844,When siblings Judy and Peter discover an encha...,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,Jumanji,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...","[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",15602,A family wedding reignites the ancient feud be...,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,Grumpier Old Men,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...","[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."
3,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",31357,"Cheated on, mistreated and stepped on, the wom...",/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,Waiting to Exhale,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...","[{'id': 818, 'name': 'based on novel'}, {'id':..."
4,"[{'id': 35, 'name': 'Comedy'}]",11862,Just when George Banks has recovered from his ...,/e64sOI48hQXyru7naBFyssKFxVd.jpg,Father of the Bride Part II,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...","[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n..."
...,...,...,...,...,...,...,...,...
46623,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",439050,Rising and falling between a man and woman.,/jldsYflnId4tTWPx8es3uzsB1I8.jpg,Subdue,"[{'cast_id': 0, 'character': '', 'credit_id': ...","[{'credit_id': '5894a97d925141426c00818c', 'de...","[{'id': 10703, 'name': 'tragic love'}]"
46624,"[{'id': 18, 'name': 'Drama'}]",111109,An artist struggles to finish his work while a...,/xZkmxsNmYXJbKVsTRLLx3pqGHx7.jpg,Century of Birthing,"[{'cast_id': 1002, 'character': 'Sister Angela...","[{'credit_id': '52fe4af1c3a36847f81e9b15', 'de...","[{'id': 2679, 'name': 'artist'}, {'id': 14531,..."
46625,"[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",67758,"When one of her hits goes wrong, a professiona...",/d5bX92nDsISNhu3ZT69uHwmfCGw.jpg,Betrayal,"[{'cast_id': 6, 'character': 'Emily Shaw', 'cr...","[{'credit_id': '52fe4776c3a368484e0c8387', 'de...",[]
46626,[],227506,"In a small town live two brothers, one a minis...",/aorBPO7ak8e8iJKT5OcqYxU3jlK.jpg,Satan Triumphant,"[{'cast_id': 2, 'character': '', 'credit_id': ...","[{'credit_id': '533bccebc3a36844cf0011a7', 'de...",[]


In [10]:
# Missing Values
msg_val = pd.DataFrame({'Missing Values' : req_movies.isnull().mean()*100, 'Data Types' : req_movies.dtypes})

# Dataframe for 2 sample values
sample = req_movies.head(2).T

# Concating
sample = pd.concat([msg_val, sample], axis = 1)

# Renaming columns
sample.rename(columns = {0: 'Sample_1', 1: 'Sample_2'}, inplace = True)

# Sample DataFrame with missing value columns
sample

,Missing Values,Data Types,Sample_1,Sample_2
genres,0.000000,object,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'id': 12, 'name': 'Adventure'}, {'id': 14, '..."
id,0.000000,object,862,8844
overview,2.133911,object,"Led by Woody, Andy's toys live happily in his ...",When siblings Judy and Peter discover an encha...
poster_path,0.855709,object,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg
title,0.008579,object,Toy Story,Jumanji
cast,0.000000,object,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'cast_id': 1, 'character': 'Alan Parrish', '..."
crew,0.000000,object,"[{'credit_id': '52fe4284c3a36847f8024f49', 'de...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
keywords,0.000000,object,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'id': 10090, 'name': 'board game'}, {'id': 1..."


In [11]:
# Dropping null values
req_movies = req_movies.dropna()

In [12]:
# Cleaned Dataset
msg_val_clean = pd.DataFrame({'Missing Values' : req_movies.isnull().mean()*100, 'Data Types' : req_movies.dtypes})
msg_val_clean

,Missing Values,Data Types
genres,0.0,object
id,0.0,object
overview,0.0,object
poster_path,0.0,object
title,0.0,object
cast,0.0,object
crew,0.0,object
keywords,0.0,object


In [13]:
# Datashape after cleaning
req_movies.shape

(45275, 8)

In [14]:
# Function to extract required names
import ast 
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name']) 
    return L 

In [15]:
# Applying the convert function on genres
req_movies['genres'] = req_movies['genres'].apply(convert)

In [16]:
# Applying the convert function on keywords
req_movies['keywords'] = req_movies['keywords'].apply(convert)

In [17]:
# Making function to find top 3 cast
def con_cast(text):
    L = []
    count = 0
    for i in ast.literal_eval(text):
        if count < 3:
            L. append(i['name'])
        count += 1
    return L

In [18]:
# Applying the con_cast function on keywords
req_movies['cast'] = req_movies['cast'].apply(con_cast)

In [19]:
# Making function to find director in crew
def director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L 

In [20]:
# Applying the director function on crew
req_movies['crew'] = req_movies['crew'].apply(director)

In [21]:
# Samples after applying diff functions
req_movies.head()

,genres,id,overview,poster_path,title,cast,crew,keywords
0,"[Animation, Comedy, Family]",862,"Led by Woody, Andy's toys live happily in his ...",/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,Toy Story,"[Tom Hanks, Tim Allen, Don Rickles]",[John Lasseter],"[jealousy, toy, boy, friendship, friends, riva..."
1,"[Adventure, Fantasy, Family]",8844,When siblings Judy and Peter discover an encha...,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,Jumanji,"[Robin Williams, Jonathan Hyde, Kirsten Dunst]",[Joe Johnston],"[board game, disappearance, based on children'..."
2,"[Romance, Comedy]",15602,A family wedding reignites the ancient feud be...,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,Grumpier Old Men,"[Walter Matthau, Jack Lemmon, Ann-Margret]",[Howard Deutch],"[fishing, best friend, duringcreditsstinger, o..."
3,"[Comedy, Drama, Romance]",31357,"Cheated on, mistreated and stepped on, the wom...",/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,Waiting to Exhale,"[Whitney Houston, Angela Bassett, Loretta Devine]",[Forest Whitaker],"[based on novel, interracial relationship, sin..."
4,[Comedy],11862,Just when George Banks has recovered from his ...,/e64sOI48hQXyru7naBFyssKFxVd.jpg,Father of the Bride Part II,"[Steve Martin, Diane Keaton, Martin Short]",[Charles Shyer],"[baby, midlife crisis, confidence, aging, daug..."


In [22]:
#Applying split on overview
req_movies['overview'] = req_movies['overview'].apply(lambda x:x.split())

In [23]:
req_movies.head()

,genres,id,overview,poster_path,title,cast,crew,keywords
0,"[Animation, Comedy, Family]",862,"[Led, by, Woody,, Andy's, toys, live, happily,...",/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,Toy Story,"[Tom Hanks, Tim Allen, Don Rickles]",[John Lasseter],"[jealousy, toy, boy, friendship, friends, riva..."
1,"[Adventure, Fantasy, Family]",8844,"[When, siblings, Judy, and, Peter, discover, a...",/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,Jumanji,"[Robin Williams, Jonathan Hyde, Kirsten Dunst]",[Joe Johnston],"[board game, disappearance, based on children'..."
2,"[Romance, Comedy]",15602,"[A, family, wedding, reignites, the, ancient, ...",/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,Grumpier Old Men,"[Walter Matthau, Jack Lemmon, Ann-Margret]",[Howard Deutch],"[fishing, best friend, duringcreditsstinger, o..."
3,"[Comedy, Drama, Romance]",31357,"[Cheated, on,, mistreated, and, stepped, on,, ...",/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,Waiting to Exhale,"[Whitney Houston, Angela Bassett, Loretta Devine]",[Forest Whitaker],"[based on novel, interracial relationship, sin..."
4,[Comedy],11862,"[Just, when, George, Banks, has, recovered, fr...",/e64sOI48hQXyru7naBFyssKFxVd.jpg,Father of the Bride Part II,"[Steve Martin, Diane Keaton, Martin Short]",[Charles Shyer],"[baby, midlife crisis, confidence, aging, daug..."


In [24]:
# Function for removing gaps between words
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

In [25]:
# Applying the function
req_movies['overview'] = req_movies['overview'].apply(collapse)
req_movies['keywords'] = req_movies['keywords'].apply(collapse)
req_movies['cast'] = req_movies['cast'].apply(collapse)
req_movies['crew'] = req_movies['crew'].apply(collapse)

In [26]:
# Making tag column with above columns
req_movies['tags'] = req_movies['overview'] + req_movies['keywords'] + req_movies['cast'] + req_movies['crew']

In [27]:
# New list of movies
new_movies = req_movies[['id', 'poster_path', 'title', 'tags']]

In [28]:
#Applying spacing between the different concatenated list of tags column
new_movies['tags'] = new_movies['tags'].apply(lambda x: " ".join(x))

In [29]:
#Applying lowercase to convert all words to lowercase so that the computer treats the same words written in different cases as identical.
new_movies['tags'] = new_movies['tags'].apply(lambda x: x.lower())

In [30]:
# Sample data of new movies
new_movies.head()

,id,poster_path,title,tags
0,862,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,Toy Story,"led by woody, andy's toys live happily in his ..."
1,8844,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,Jumanji,when siblings judy and peter discover an encha...
2,15602,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,Grumpier Old Men,a family wedding reignites the ancient feud be...
3,31357,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,Waiting to Exhale,"cheated on, mistreated and stepped on, the wom..."
4,11862,/e64sOI48hQXyru7naBFyssKFxVd.jpg,Father of the Bride Part II,just when george banks has recovered from his ...


In [31]:
# Shape of new_movies
new_movies.shape

(45275, 4)

In [32]:
#Vectorizer
vectorizer = TfidfVectorizer(max_features = 10000, stop_words = 'english')

In [33]:
#Fit transforming tags column to convert the words into vectors
vector = vectorizer.fit_transform(new_movies['tags']).toarray()

In [34]:
#List of top 10,000 most appearing words alphabetically arranged
list(vectorizer.get_feature_names_out()[:100])

['000',
 '10',
 '100',
 '11',
 '12',
 '12th',
 '13',
 '13th',
 '14',
 '15',
 '150',
 '16',
 '16th',
 '17',
 '17th',
 '18',
 '18th',
 '19',
 '1912',
 '1915',
 '1917',
 '1918',
 '1920',
 '1920s',
 '1930',
 '1930s',
 '1931',
 '1934',
 '1936',
 '1937',
 '1938',
 '1939',
 '1940',
 '1940s',
 '1941',
 '1942',
 '1943',
 '1944',
 '1945',
 '1946',
 '1947',
 '1948',
 '1949',
 '1950',
 '1950s',
 '1951',
 '1952',
 '1953',
 '1954',
 '1955',
 '1956',
 '1957',
 '1958',
 '1959',
 '1960',
 '1960s',
 '1961',
 '1962',
 '1963',
 '1964',
 '1965',
 '1966',
 '1967',
 '1968',
 '1969',
 '1970',
 '1970s',
 '1971',
 '1972',
 '1973',
 '1974',
 '1975',
 '1976',
 '1977',
 '1978',
 '1979',
 '1980',
 '1980s',
 '1981',
 '1982',
 '1983',
 '1984',
 '1985',
 '1986',
 '1987',
 '1988',
 '1989',
 '1990',
 '1990s',
 '1991',
 '1992',
 '1993',
 '1994',
 '1995',
 '1996',
 '1997',
 '1998',
 '1999',
 '19th',
 '19thcentury']

In [35]:
#Applying stemming 
ps = PorterStemmer()
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

new_movies['tags'] = new_movies['tags'].apply(stem)

In [36]:
#Vectorizer
vectorizer = TfidfVectorizer(max_features = 10000, stop_words = 'english')

#Fit transforming tags column to convert the words into vectors
vector = vectorizer.fit_transform(new_movies['tags']).toarray()

#List of top 5000 most appearing words alphabetically arranged
list(vectorizer.get_feature_names_out()[:200])

['000',
 '10',
 '100',
 '1000',
 '11',
 '12',
 '12th',
 '13',
 '13th',
 '14',
 '15',
 '150',
 '15th',
 '16',
 '16th',
 '17',
 '17th',
 '18',
 '1890',
 '18th',
 '18thcenturi',
 '19',
 '1900',
 '1910',
 '1912',
 '1914',
 '1915',
 '1917',
 '1918',
 '1920',
 '1920s',
 '1927',
 '1930',
 '1930s',
 '1931',
 '1932',
 '1933',
 '1934',
 '1935',
 '1936',
 '1937',
 '1938',
 '1939',
 '1940',
 '1940s',
 '1941',
 '1942',
 '1943',
 '1944',
 '1945',
 '1946',
 '1947',
 '1948',
 '1949',
 '1950',
 '1950s',
 '1951',
 '1952',
 '1953',
 '1954',
 '1955',
 '1956',
 '1957',
 '1958',
 '1959',
 '1960',
 '1960s',
 '1961',
 '1962',
 '1963',
 '1964',
 '1965',
 '1966',
 '1967',
 '1968',
 '1969',
 '1970',
 '1970s',
 '1971',
 '1972',
 '1973',
 '1974',
 '1975',
 '1976',
 '1977',
 '1978',
 '1979',
 '1980',
 '1980s',
 '1981',
 '1982',
 '1983',
 '1984',
 '1985',
 '1986',
 '1987',
 '1988',
 '1989',
 '1990',
 '1990s',
 '1991',
 '1992',
 '1993',
 '1994',
 '1995',
 '1996',
 '1997',
 '1998',
 '1999',
 '19th',
 '19thcenturi',
 '

In [50]:
def recommend(movie):
    if movie not in new_movies['title'].values:
        print("Movie not found in dataset.")
        return

    movie_index = new_movies[new_movies['title'] == movie].index[0]
    sim_scores = cosine_similarity(vector[movie_index].reshape(1, -1), vector).flatten()

    # Sort and pick top 5 most similar (excluding the movie itself)
    similar_movies = sorted(list(enumerate(sim_scores)), key=lambda x: x[1], reverse=True)[1:6]

    print(f"\nMovies similar to '{movie}':\n")
    for i, score in similar_movies:
        print(f"{new_movies.iloc[i].title:<40} Similarity Score: {score:.4f}")

In [52]:
recommend('Batman Begins')


Movies similar to 'Batman Begins':

Thank Your Lucky Stars                   Similarity Score: 0.2138
The Lookalike                            Similarity Score: 0.2082
Bruno                                    Similarity Score: 0.1772
Bullet to Beijing                        Similarity Score: 0.1750
Luv                                      Similarity Score: 0.1724
